# 2.2 Streaming and structured data

Step 2, lesson 2 of [Building with the Claude API](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287734) (Anthropic Academy). The lesson ends with the API quiz, which is not reproduced here.
Exam domain: **4 Prompt Engineering & Structured Output** (getting reliable JSON out of Claude). Streaming is **out of exam scope**.

What this notebook covers, in the course's order:

1. Streaming: raw events, and the `messages.stream()` helper
2. Controlling the output: prefill (gone on current models) and stop sequences
3. Structured data the course's way, and the closest thing that still works
4. Structured outputs: `output_config.format` with a JSON schema
5. `client.messages.parse()` with a Pydantic model, and what a schema does *not* guarantee

Three exercises along the way let you practise each part. For each exercise: write your code in the *Your turn* cell and run it, then run the *Check* cell. The *Solution* cell sits in between: open the hint first, and skip the solution unless you're stuck.

> **Model used here:** `claude-sonnet-5-5`. Small calls use `output_config={"effort": "low"}` to keep them cheap (see 2.1).

In [1]:
import json
from collections import Counter

import anthropic
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-sonnet-5-5"


def text_of(message):
    """Return the text of the first text block in a response ("" if there is none)."""
    return next((block.text for block in message.content if block.type == "text"), "")

## 1. Streaming

A normal call returns only when the whole answer is written, which can take many seconds for a long reply. With **streaming**, the API sends the answer as a series of small *events* while Claude is still writing, so a chat app can show the text as it appears.

> **Out of exam scope.** The exam guide lists streaming as not examined. One look at how it works, then on to structured data, which *is* examined.

The low-level way is `stream=True`: you get the raw server-sent events. The ones to recognise:

| Event | Carries |
|---|---|
| `message_start` | the empty message (id, model, input token count) |
| `content_block_start` / `content_block_stop` | the start and end of each block (text, thinking, tool use) |
| `content_block_delta` | a small piece of a block, e.g. a few words of text (`text_delta`) |
| `message_delta` | the `stop_reason` and the final output token count |
| `message_stop` | the end |

In [2]:
stream = client.messages.create(
    model=model,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Write a two-sentence description of a fictional database product."}],
    stream=True,
)

event_types = []
for event in stream:
    event_types.append(event.type)
    if event.type == "content_block_delta" and event.delta.type == "text_delta":
        print(event.delta.text, end="", flush=True)

print("\n")
print(Counter(event_types))

**Strata

D

B** is a dist

ributed,

 time-a

ware database that stores every version

... (21 outputs left out)

 older

 lay

ers.



Counter({'content_block_delta': 30, 'message_start': 1, 'content_block_start': 1, 'content_block_stop': 1, 'message_delta': 1, 'message_stop': 1})


The SDK's helper `client.messages.stream(...)` does the bookkeeping: `text_stream` yields only the text pieces, and `get_final_message()` gives the assembled `Message` at the end, the same object a normal call returns. That last part is what you store, e.g. to add the reply to the conversation history.

In [3]:
with client.messages.stream(
    model=model,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Write a two-sentence description of a fictional cloud storage product."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()

print("\n")
print(final.stop_reason, "|", final.usage.output_tokens, "output tokens")

**S

kyV

ault** is a cloud stor

age platform that autom

atically organ

izes your files using

... (21 outputs left out)

 private and protected from accidental

 chang

es.



end_turn | 112 output tokens


### Exercise 1 · How much sooner does streaming show something?

Streaming doesn't make the answer *finish* sooner; it makes the first words appear sooner. Measure both.

**Task.** Stream an answer of about 150 words (any topic) with `client.messages.stream(...)`. Set `first` to the seconds until the first piece of text arrived, and `total` to the seconds until the end.

**Done when** the check cell prints ✅.

In [4]:
# Your turn
import time

first = total = None
start = time.perf_counter()
# stream here: set `first` when the first text arrives, and `total` at the end

<details><summary><b>Hint</b></summary>

Copy the `messages.stream` cell above. Inside `for text in stream.text_stream:`, set `first = time.perf_counter() - start` only if `first is None`. After the `with` block, set `total` the same way.

</details>

**Solution** (skip it unless you're stuck)

In [5]:
# Solution
start = time.perf_counter()
first = None
with client.messages.stream(
    model=model,
    max_tokens=600,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "In about 150 words, explain why the sky is blue."}],
) as stream:
    for text in stream.text_stream:
        if first is None:
            first = time.perf_counter() - start
        print(text, end="", flush=True)
total = time.perf_counter() - start

The sky loo

ks blue because of

 how

 s

unlight interacts with

 Earth's atmosphere

... (61 outputs left out)

 atmosphere, so most

 blue is scattered away

, leaving reds and o

ranges.

In [6]:
# Check
if first is not None and total is not None and first < total:
    print(f"✅ First text after {first:.1f} s, the whole answer after {total:.1f} s. "
          f"Without streaming, the screen would stay blank for {total:.1f} s.")
else:
    print("❌ Not yet: set first (when the first text arrives) and total (at the end).")

✅ First text after 1.0 s, the whole answer after 3.5 s. Without streaming, the screen would stay blank for 3.5 s.


## 2. Controlling the output: prefill and stop sequences

The course shows two ways to steer *where an answer starts and ends*:

- **Prefill**: end `messages` with a partial `assistant` message; Claude continues from those words.
- **Stop sequences**: a list of strings; generation stops as soon as Claude writes one of them.

> **Changed since the course.** On current models (Opus 5.5, Sonnet 5.5) a request that ends with an assistant message is rejected with a 400. Steer the start of the answer with instructions instead, or use structured outputs (section 4) when what you wanted was clean JSON.

In [7]:
try:
    client.messages.create(
        model=model,
        max_tokens=100,
        messages=[
            {"role": "user", "content": "Is coffee or tea better before a morning run? Pick one."},
            {"role": "assistant", "content": "Coffee is better because"},  # the prefill
        ],
    )
except anthropic.BadRequestError as e:
    print("API 400:", e.message)

API 400: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011Ch67UvYfhbhro3v9a77kv'}


Stop sequences still work. The text stops *before* the stop sequence (it is not included), `stop_reason` is `"stop_sequence"` and `stop_sequence` says which one was hit:

In [8]:
message = client.messages.create(
    model=model,
    max_tokens=200,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas. Output only the numbers."}],
    stop_sequences=["5"],
)
print(repr(text_of(message)))
print(message.stop_reason, repr(message.stop_sequence))

'1, 2, 3, 4, '
stop_sequence '5'


### Exercise 2 · Stop after three

**Task.** Ask for a numbered list of five names for a study app, one per line (`1. ...`, `2. ...`), and use a stop sequence so that only the first three come back. Keep the response in `three`.

**Done when** the check cell prints ✅.

In [9]:
# Your turn
three = None  # client.messages.create(..., stop_sequences=[...])

<details><summary><b>Hint</b></summary>

The fourth item starts with `4.` at the beginning of a new line, so `"\n4."` is a safe stop sequence (a bare `"4"` could also match inside a name). The stop text itself is not included in the answer.

</details>

**Solution** (skip it unless you're stuck)

In [10]:
# Solution
three = client.messages.create(
    model=model,
    max_tokens=300,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": "List five names for a study app, as a numbered list, one per line. Output only the list."}],
    stop_sequences=["\n4."],
)
print(text_of(three))

1. StudySpark
2. BrainBloom
3. FocusFlow


In [11]:
# Check
items = [ln for ln in text_of(three).splitlines() if ln.strip()] if three is not None else []
if three is not None and three.stop_reason == "stop_sequence" and len(items) == 3:
    print(f"✅ Stopped at {three.stop_sequence!r}, with three items.")
else:
    print("❌ Not yet: the response should stop at your stop sequence, with exactly three items.")

✅ Stopped at '\n4.', with three items.


## 3. Structured data, the course's way

Asked for JSON, Claude tends to add a friendly sentence and wrap the JSON in a Markdown code fence. That is nice in a chat window and annoying in code that calls `json.loads`. The course's example generates an AWS EventBridge rule, and its trick was to **prefill** the answer with an opening ```` ```json ```` fence and set a **stop sequence** on the closing ```` ``` ````, so the reply contained only the raw JSON.

Without prefill, the nearest working version asks for the JSON inside tags and stops at the closing tag. Then you keep what comes after the opening tag:

In [12]:
message = client.messages.create(
    model=model,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": (
        "Generate a very short AWS EventBridge event pattern, as JSON, that matches EC2 instances "
        "changing to the 'stopped' state. Write only the JSON, without code fences, inside <json></json> tags."
    )}],
    stop_sequences=["</json>"],
)
raw = text_of(message)
print("stop:", message.stop_reason, "| raw starts with:", repr(raw[:40]))

pattern = json.loads(raw.split("<json>", 1)[1])
print(json.dumps(pattern, indent=2))

stop: stop_sequence | raw starts with: '<json>{"source":["aws.ec2"],"detail-type'
{
  "source": [
    "aws.ec2"
  ],
  "detail-type": [
    "EC2 Instance State-change Notification"
  ],
  "detail": {
    "state": [
      "stopped"
    ]
  }
}


This works, but nothing *guarantees* it: Claude could still add a fence inside the tags, forget a closing bracket, or invent a field name. For anything a program depends on, use structured outputs.

## 4. Structured outputs: `output_config.format`

> **Beyond the course.** The course predates this feature. It is the current, recommended way to get JSON.

You pass a JSON schema, and the API **constrains generation** so the reply is valid JSON that matches it: right field names, right types, required fields present. The rules for the schema:

- every object sets `"additionalProperties": false` and lists its `required` fields;
- numeric and length limits (`minimum`, `maxLength`, ...) are not supported, so check those in code.

The example is the kind of task Domain 4 asks about: extracting fields from a customer email. Two schema choices matter:

- **Nullable fields** (`anyOf` string or null): if the email has no order number, the model has a legitimate "not present" answer instead of being pushed to invent one.
- **An enum with an `"other"` value plus a detail field**: real inputs don't always fit your categories.

In [13]:
ticket_schema = {
    "type": "object",
    "properties": {
        "customer_name": {"type": "string"},
        "order_id": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "null if the email gives no order number",
        },
        "category": {"type": "string", "enum": ["billing", "shipping", "damaged_item", "technical", "other"]},
        "category_detail": {
            "anyOf": [{"type": "string"}, {"type": "null"}],
            "description": "Short explanation, only when category is 'other'; otherwise null",
        },
        "urgent": {"type": "boolean"},
        "summary": {"type": "string", "description": "One sentence"},
    },
    "required": ["customer_name", "order_id", "category", "category_detail", "urgent", "summary"],
    "additionalProperties": False,
}

email = """Hi, this is Dana Whitfield. The blender I bought arrived with a cracked jug,
and I need it working before my daughter's party on Saturday. I can't find my order
number, sorry! Can you send a replacement?"""

message = client.messages.create(
    model=model,
    max_tokens=1000,
    output_config={
        "effort": "low",
        "format": {"type": "json_schema", "schema": ticket_schema},
    },
    messages=[{"role": "user", "content": f"Extract the support ticket fields from this email.\n\n<email>\n{email}\n</email>"}],
)
ticket = json.loads(text_of(message))   # no try/except needed: the format is guaranteed
print(json.dumps(ticket, indent=2))

{
  "customer_name": "Dana Whitfield",
  "order_id": null,
  "category": "damaged_item",
  "category_detail": null,
  "urgent": true,
  "summary": "Dana Whitfield received a blender with a cracked jug and needs a replacement before her daughter's party on Saturday."
}


## 5. `client.messages.parse()` with a Pydantic model

Writing JSON schemas by hand gets tedious. The Python SDK can build the schema from a **Pydantic** model: pass the class as `output_format` to `client.messages.parse(...)` and read a validated object from `response.parsed_output`. (The SDK turns it into `output_config.format` for you.)

The example extracts an invoice. On purpose, the printed total does not match the line items.

In [14]:
from pydantic import BaseModel, Field


class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    vendor: str
    invoice_number: str | None = Field(description="null if no invoice number is printed")
    line_items: list[LineItem]
    stated_total: float = Field(description="The total exactly as printed on the invoice")


invoice_text = """ACME Office Supplies, Invoice A-1043
3 x Printer paper (box) @ 25.00
2 x Ink cartridge @ 40.00
TOTAL DUE: 145.00"""

response = client.messages.parse(
    model=model,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": f"Extract this invoice.\n\n<invoice>\n{invoice_text}\n</invoice>"}],
    output_format=Invoice,
)
invoice = response.parsed_output
print(invoice.model_dump_json(indent=2))

{
  "vendor": "ACME Office Supplies",
  "invoice_number": "A-1043",
  "line_items": [
    {
      "description": "Printer paper (box)",
      "quantity": 3,
      "unit_price": 25.0
    },
    {
      "description": "Ink cartridge",
      "quantity": 2,
      "unit_price": 40.0
    }
  ],
  "stated_total": 145.0
}


The JSON is valid and matches the schema. Is it *right*? A schema checks shape, not meaning, so the arithmetic is checked in code:

In [15]:
computed = sum(item.quantity * item.unit_price for item in invoice.line_items)
print(f"line items add up to {computed:.2f}; the invoice says {invoice.stated_total:.2f}")

if abs(computed - invoice.stated_total) > 0.005:
    print("Mismatch: flag this invoice for human review.")

line items add up to 155.00; the invoice says 145.00
Mismatch: flag this invoice for human review.


**Exam angle (Domain 4).** Structured outputs (and strict tool schemas, lesson 2.6) remove *syntax* errors: no broken JSON, no missing required fields. They don't remove *semantic* errors: totals that don't add up, a value put in the wrong field, a plausible but invented value. Two follow-ups, depending on the cause:

- the model made the mistake → a **validation-retry loop**: send the specific error back ("line items sum to 155.00 but stated_total is 145.00, re-check the document") and ask again;
- the source itself is inconsistent or the information is missing → retrying won't help; return null or flag the record for a human.

Design the schema to help: nullable fields for information that may be absent, `"other"` plus a detail field in enums, and a field such as `stated_total` next to the computed one, so the check can be done in code.

### Exercise 3 · Extract a meeting, then check what the schema can't

This is the course's structured data exercise, done the current way.

**Task.**

1. Write Pydantic models `ActionItem` (`owner`, `task`) and `Meeting` (`title`, `date` as `YYYY-MM-DD`, `attendees`, `action_items`).
2. Extract `meeting_note` into a `Meeting` with `client.messages.parse(...)`, and keep it in `meeting`.
3. A schema can't know that every task should belong to someone who was at the meeting. Check that in code: put the owners who are *not* attendees in `unknown_owners`.

**Done when** the check cell prints ✅. The note is written so that one owner wasn't there.

In [16]:
# Your turn
meeting_note = """Sprint review, 14 November 2026. Present: Priya, Tom and Lena.
Tom will fix the login timeout before Friday. Lena is writing the release notes.
Priya asked Marco, who couldn't make it, to update the API docs."""

meeting = None          # client.messages.parse(..., output_format=Meeting).parsed_output
unknown_owners = None   # owners who are not in meeting.attendees

<details><summary><b>Hint</b></summary>

Follow the invoice example: one small model per kind of thing, a `list[ActionItem]` field, and a `Field(description=...)` where the format matters ("YYYY-MM-DD", "only the people who were present"). The check is one list comprehension over `meeting.action_items`.

</details>

**Solution** (skip it unless you're stuck)

In [17]:
# Solution
class ActionItem(BaseModel):
    owner: str
    task: str


class Meeting(BaseModel):
    title: str
    date: str = Field(description="YYYY-MM-DD")
    attendees: list[str] = Field(description="Only the people who were present")
    action_items: list[ActionItem]


response = client.messages.parse(
    model=model,
    max_tokens=1000,
    output_config={"effort": "low"},
    messages=[{"role": "user", "content": f"Extract this meeting note.\n\n<note>\n{meeting_note}\n</note>"}],
    output_format=Meeting,
)
meeting = response.parsed_output
print(meeting.model_dump_json(indent=2))

unknown_owners = [item.owner for item in meeting.action_items if item.owner not in meeting.attendees]
print("Owners who weren't there:", unknown_owners)

{
  "title": "Sprint review",
  "date": "2026-11-14",
  "attendees": [
    "Priya",
    "Tom",
    "Lena"
  ],
  "action_items": [
    {
      "owner": "Tom",
      "task": "Fix the login timeout before Friday"
    },
    {
      "owner": "Lena",
      "task": "Write the release notes"
    },
    {
      "owner": "Marco",
      "task": "Update the API docs"
    }
  ]
}
Owners who weren't there: ['Marco']


In [18]:
# Check
if meeting is not None and unknown_owners == ["Marco"]:
    print("✅ Valid JSON, and the check in code caught what the schema couldn't: Marco owns a task but wasn't there.")
else:
    print("❌ Not yet: extract the meeting, then list the action item owners who are not attendees.")

✅ Valid JSON, and the check in code caught what the schema couldn't: Marco owns a task but wasn't there.


## Recap

- Streaming sends the reply as events (`text_stream`, `get_final_message()`). Useful for UIs, **not examined**.
- Prefill returns a 400 on current models; stop sequences still work (the stop text is not included).
- The course's prefill + stop sequence trick for JSON is replaced by **structured outputs**: `output_config.format` with a JSON schema, or `client.messages.parse(output_format=PydanticModel)`.
- A schema guarantees shape, not truth: validate meaning in code, retry with the error when the model slipped, flag for review when the source is the problem (Domain 4).

Next: [2.3 Prompt evaluation](03-prompt-evaluation.ipynb).